# Epigenetic marks: image-model predictions per cell (inference kernel)

**Models:**
1. **ImmuVis trained on Jackson with H3K27me3 and pHH3** (`epi-immuvis-train`). Both marks are predicted from the 14 shared
   channels (H3, DNA1, DNA2 + 11 shared markers), the only inputs Danenberg has.
   - **Leave-anchor-out (LOA):** the same model, given the 14 channels minus one shared marker M (11 runs). Relations between a
     mark and M can then be checked with predictions that never saw M (the circularity control in `epi-relations`).
2. **EVA, zero-shot.** EVA names markers through `marker_to_gene`; H3K27me3, pHH3 (`HistoneH3p`) and total H3 (`HistoneH3`)
   all map to the gene H3F3A. This kernel records whether that gives them the same token, and predicts both marks anyway.
   EVA scales every channel per image, so its outputs are used only within images.

**Cells:** the same analysis cells as `epi-relations` (tumour images with ≥ 200 cells).
- Jackson: the test patients of replicate 0 (never seen in training or checkpoint selection).
- Danenberg: every image (no Danenberg data were used in training). LOA runs on a fixed random subset of images to save GPU time.

**Output:** per-cell means of the pixel predictions under the published masks (`jackson-cells-prep`, `danenberg-cells-prep`):
`epi_pred_jackson_test.parquet`, `epi_pred_danenberg.parquet`; pixel sums for the Jackson test images; `run_info.json`.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SEED = 20261013
MIN_CELLS = 200
COF = 5.0
N_LOA_DANENBERG = int(os.environ.get("N_LOA_DANENBERG", 300))
NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)   # image name 'c_erb_b_2_her2' = clone 3B5 in both
EPI = ["histone_h3_trimethylate", "histone_h3_phospho"]
SHORT = {"histone_h3_trimethylate": "h3k27me3", "histone_h3_phospho": "phh3"}
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()
FAILED = []

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

## Analysis cells (same filters and order as `epi-relations`)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"

P = {}
for p in g("cells.parquet"):
    P["danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"] = p

def load_cells(key):
    if key == "danenberg":
        c = pd.read_parquet(P[key], columns=["image_name", "object_number", "metabric_id", "is_tumour", "is_hot_aggregate"])
        c = c[(c.is_tumour == 1) & (c.is_hot_aggregate == 0)]
        c["patient"] = c.metabric_id.astype(str)
    else:
        c = pd.read_parquet(P[key], columns=["image_name", "object_number", "patient_id", "is_tumour"])
        c = c[c.is_tumour == 1]
        c["patient"] = c.patient_id.astype(str)
    n_per = c.groupby("image_name").size()
    c = c[c.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    return c.sort_values(["image_name", "object_number"]).reset_index(drop=True)[["image_name", "object_number", "patient"]]

CJ, CD = load_cells("jackson"), load_cells("danenberg")
jp, jcode = np.unique(CJ.patient.values, return_inverse=True)
_u = np.random.default_rng(SEED + 11 + 0).random(len(jp))
CJ["split"] = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[jcode]
CJT = CJ[CJ.split == "test"].reset_index(drop=True)

tifs = g("*.tiff")
def _is_mask(p):
    return "/masks/" in p.replace("\\", "/")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if os.path.basename(p).startswith("MB_") and not _is_mask(p)}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if os.path.basename(p).startswith("MB_") and _is_mask(p)}
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and not _is_mask(p) and "jackson" in p.lower()}
J_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if not os.path.basename(p).startswith("MB_") and _is_mask(p)}
CJT = CJT[CJT.image_name.isin(set(J_IMG) & set(J_MASK))].reset_index(drop=True)
CD = CD[CD.image_name.isin(set(D_IMG) & set(D_MASK))].reset_index(drop=True)
J_TEST = list(dict.fromkeys(CJT.image_name))
D_ALL = list(dict.fromkeys(CD.image_name))
D_LOA = set(np.random.default_rng(SEED + 31).choice(D_ALL, min(N_LOA_DANENBERG, len(D_ALL)), replace=False))
print(f"Jackson test: {len(CJT):,} cells, {len(J_TEST)} images, {CJT.patient.nunique()} patients")
print(f"Danenberg: {len(CD):,} cells, {len(D_ALL)} images, {CD.patient.nunique()} patients; LOA on {len(D_LOA)} images")
dix = {m: i for i, m in enumerate(D_CHAN)}
jix = {m: i for i, m in enumerate(J_CHAN)}

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def cell_lookup(mask, obj):
    """Pixel -> position of the cell in obj (or -1), for one image."""
    lut = np.full(max(int(mask.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    return lut[mask].ravel()

def spans(cells):
    """image_name -> (first row, end row); cells are sorted by image_name."""
    s = cells.image_name.values
    st = np.r_[0, np.flatnonzero(s[1:] != s[:-1]) + 1, len(s)]
    return {s[a]: (a, b) for a, b in zip(st[:-1], st[1:])}

def cell_means(loc, n, maps):
    v = loc >= 0
    cnt = np.bincount(loc[v], minlength=n)
    got = cnt > 0
    out = {}
    for k, M in maps.items():
        s = np.bincount(loc[v], M.ravel()[v].astype(np.float64), n)
        out[k] = np.where(got, s / np.maximum(cnt, 1), np.nan).astype(np.float32)
    return out, cnt

## ImmuVis (16 tokens, trained on Jackson)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request
from torch.amp import autocast
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
sys.path.insert(0, root)
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import EncoderConfig, DecoderConfig
torch.backends.cudnn.enabled = False        # immuvis-profile: cuDNN on a T4 is slow and leaks host RAM for this model

CH = NUC + MARKERS + EPI
SH = list(range(len(NUC) + len(MARKERS)))
EP = [CH.index(e) for e in EPI]
IDS = torch.arange(len(CH))
CLIP = 2.3
CK = torch.load(g("epi_immuvis_jackson_best.pth")[0], map_location="cpu", weights_only=False)
assert CK["channels"] == CH and CK["clip"] == CLIP and CK["train_cohort"] == "jackson", (CK["channels"], CH)
ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4], pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1,
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
net = MultiplexAutoencoder(num_channels=len(CH), encoder_config=ENC, decoder_config=DEC).to(dev)
net.load_state_dict(CK["model_state_dict"])
net.eval()
print(f"checkpoint: step {CK['step']}/{CK['total_steps']}, val pixel r {CK['val_r_at_best']}, {CK['n_train_images']} train images")

T_IN, M_CTX = 128, 32
SUBSETS = {"full": SH, **{f"loa_{m}": [i for i in SH if CH[i] != m] for m in MARKERS}}

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

@torch.no_grad()
def immuvis_epi(x, subsets):
    """x: [14, H, W] preprocessed shared channels. Returns {subset: [2, H, W]} maps of (H3K27me3, pHH3)."""
    C, H, W = x.shape
    tiles = [(y0, x0) for y0 in range(0, H, T_IN) for x0 in range(0, W, T_IN)]
    win = T_IN + 2 * M_CTX
    xp = np.pad(x, ((0, 0), (M_CTX, M_CTX + T_IN), (M_CTX, M_CTX + T_IN)))
    batch = torch.from_numpy(np.stack([xp[:, y0:y0 + win, x0:x0 + win] for y0, x0 in tiles]).astype(np.float32))
    res = {}
    for name in subsets:
        act = SUBSETS[name]
        outs = []
        for b0 in range(0, len(batch), 64):
            xb = batch[b0:b0 + 64].to(dev)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lat = net.encode(xb[:, act], IDS[act].to(dev).expand(len(xb), -1))["output"]
                o = net.decode(lat, IDS[EP].to(dev).expand(len(xb), -1))
            outs.append(torch.sigmoid(o[..., 0].float())[:, :, M_CTX:M_CTX + T_IN, M_CTX:M_CTX + T_IN].cpu().numpy())
        outs = np.concatenate(outs)
        Pm = np.zeros((len(EP), H + T_IN, W + T_IN), np.float32)
        for (y0, x0), o in zip(tiles, outs):
            Pm[:, y0:y0 + T_IN, x0:x0 + T_IN] = o
        res[name] = Pm[:, :H, :W]
    return res

def run_cohort(cohort, cells, names, img_map, mask_map, n_chan, idx, loa_names):
    """Per-cell ImmuVis predictions for every image in names; LOA subsets only for images in loa_names."""
    cols = {f"immuvis__{s}__{SHORT[e]}": np.full(len(cells), np.nan, np.float32) for s in SUBSETS for e in EPI}
    npix = np.zeros(len(cells), np.int32)
    pix = []
    span = spans(cells)
    for j, s in enumerate(names):
        a, b = span[s]
        im = read_hwc(img_map[s], n_chan)
        mask = tifffile.imread(mask_map[s]).astype(np.int64)
        assert mask.shape == im.shape[:2], (s, mask.shape, im.shape)
        x = immuvis_pre(im, NUC + MARKERS, idx)
        subs = list(SUBSETS) if s in loa_names else ["full"]
        res = immuvis_epi(x, subs)
        loc = cell_lookup(mask, cells.object_number.values[a:b].astype(np.int64))
        maps = {f"immuvis__{sn}__{SHORT[e]}": res[sn][k] for sn in subs for k, e in enumerate(EPI)}
        cm, cnt = cell_means(loc, b - a, maps)
        for k_, v in cm.items():
            cols[k_][a:b] = v
        npix[a:b] = cnt
        if cohort == "jackson":     # pixel sums vs the measured marks in ImmuVis space (low-pass, clipped, / 2.3)
            truth = immuvis_pre(im, EPI, idx)
            for k, e in enumerate(EPI):
                G = truth[k].astype(np.float64).ravel()
                Pf = res["full"][k].astype(np.float64).ravel()
                pix.append(dict(image=s, mark=SHORT[e], n=G.size, Sg=G.sum(), Sgg=G @ G, Sp=Pf.sum(), Spp=Pf @ Pf, Sgp=G @ Pf))
        if j % 25 == 0:
            tick(f"ImmuVis {cohort}: {j + 1}/{len(names)} images")
    out = cells[["image_name", "object_number", "patient"]].copy()
    out["n_pixels"] = npix
    for k_, v in cols.items():
        out[k_] = v
    return out, pd.DataFrame(pix)

RUN = {"immuvis_checkpoint": {k: CK[k] for k in ["step", "total_steps", "best_val_epi_pixel_r", "val_r_at_best",
                                                   "train_minutes", "n_train_images", "n_val_images", "code_source"]}}
PJ = PD = None
try:
    PJ, PIX = run_cohort("jackson", CJT, J_TEST, J_IMG, J_MASK, len(J_CHAN), jix, set(J_TEST))
    PIX.to_csv(f"{OUT}/pixel_sums_jackson_test_immuvis.csv", index=False)
    for mk, d in PIX.groupby("mark"):
        n, Sg, Sp, Sgg, Spp, Sgp = (d[c].values for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"])
        with np.errstate(invalid="ignore", divide="ignore"):
            per = (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))
        N_, G_, P_ = n.sum(), Sg.sum(), Sp.sum()
        pooled = (Sgp.sum() - G_ * P_ / N_) / np.sqrt((Sgg.sum() - G_ ** 2 / N_) * (Spp.sum() - P_ ** 2 / N_))
        RUN[f"pixel_r_jackson_test_{mk}"] = dict(pooled=float(pooled), per_image_mean=float(np.nanmean(per)))
    print({k: v for k, v in RUN.items() if k.startswith("pixel")})
    PJ.to_parquet(f"{OUT}/epi_pred_jackson_test.parquet", index=False)      # rewritten with EVA columns below
    tick("ImmuVis Jackson test done")
except Exception:
    print("!!! ImmuVis Jackson failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("immuvis_jackson")
try:
    PD, _ = run_cohort("danenberg", CD, D_ALL, D_IMG, D_MASK, len(D_CHAN), dix, D_LOA)
    tick("ImmuVis Danenberg done")
except Exception:
    print("!!! ImmuVis Danenberg failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("immuvis_danenberg")
if PD is not None:
    PD.to_parquet(f"{OUT}/epi_pred_danenberg.parquet", index=False)
del net
if dev == "cuda":
    torch.cuda.empty_cache()
torch.backends.cudnn.enabled = True      # cuDNN was off only for ImmuVis

## EVA, zero-shot (Jackson test images; within-image use only)

In [ ]:
EVA_INFO = {}
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "einops"], check=True)
    EVA_DIR = "/tmp/Eva"
    if not os.path.isdir(EVA_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/YAndrewL/Eva.git", EVA_DIR], check=True)
    EVA_COMMIT = subprocess.run(["git", "-C", EVA_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    os.makedirs(f"{EVA_DIR}/marker_embeddings", exist_ok=True)
    if not os.path.exists(f"{EVA_DIR}/marker_embeddings/GenePT_embedding.pkl"):
        os.symlink(g("GenePT_embedding.pkl")[0], f"{EVA_DIR}/marker_embeddings/GenePT_embedding.pkl")
    os.chdir(EVA_DIR)
    sys.path.insert(0, EVA_DIR)
    from omegaconf import OmegaConf
    from Eva.eva import EvaMAE
    from Eva.layers import MarkerEmbeddingGenePT
    from utils.constant import marker_to_gene
    eva = EvaMAE.from_checkpoint(g("Eva_model.ckpt")[0], OmegaConf.load(f"{EVA_DIR}/config.yaml"), device=dev)
    eva.eval()
    os.chdir(OUT)
    EVA_NAME = {"histone_h3": "HistoneH3", "c_erb_b_2_her2": "HER2", "cd20": "CD20", "cd3": "CD3e", "cd45": "CD45",
                "cd68": "CD68", "cytokeratin_5": "Keratin5", "cytokeratin_8_18": "Keratin8/18",
                "estrogen_receptor_alpha": "ERa", "ki_67": "Ki67", "pan_cytokeratin": "PanCK", "sma": "aSMA"}
    EVA_EPI = {"histone_h3_trimethylate": "H3K27me3", "histone_h3_phospho": "HistoneH3p"}
    ME = next(m for m in eva.modules() if isinstance(m, MarkerEmbeddingGenePT))
    with torch.no_grad():
        E3 = ME(["H3K27me3", "HistoneH3p", "HistoneH3"]).float().cpu()
    EVA_INFO = dict(commit=EVA_COMMIT, genes={k: marker_to_gene[k] for k in ["H3K27me3", "HistoneH3p", "HistoneH3"]},
                    h3f3a_has_genept_embedding=bool(marker_to_gene["H3K27me3"] in ME.genept_embeddings),
                    token_identical_h3k27me3_phh3=bool(torch.equal(E3[0], E3[1])),
                    token_identical_h3k27me3_h3=bool(torch.equal(E3[0], E3[2])),
                    token_cosine_h3k27me3_h3=float(torch.nn.functional.cosine_similarity(E3[0], E3[2], dim=0)))
    print("EVA tokens:", EVA_INFO)
    CH_EVA = ["DAPI"] + [EVA_NAME[c] for c in ["histone_h3"] + MARKERS]       # 13 inputs = the 14 shared channels
    TILE, TOK = 224, 8
    NTOK = (TILE // TOK) ** 2

    def eva_channels(im_hwc, idx):
        """[H, W, 13] in [0, 1]: arcsinh then per-image [p0.5, p99.5] (the scaling chosen in eva-zeroshot)."""
        raw = [0.5 * (im_hwc[..., idx["dna1"]] + im_hwc[..., idx["dna2"]])] + [im_hwc[..., idx[c]] for c in ["histone_h3"] + MARKERS]
        out = []
        for c in raw:
            c = np.arcsinh(c / COF)
            lo, hi = np.percentile(c, [0.5, 99.5])
            out.append(np.clip((c - lo) / (hi - lo + 1e-8), 0, 1))
        return np.stack(out, -1).astype(np.float32)

    @torch.no_grad()
    def eva_extra(x, target):
        """Predict an unmeasured channel (token `target`) from the 13 inputs: an extra all-zero, fully masked channel."""
        H, W, C = x.shape
        Hp, Wp = -(-H // TILE) * TILE, -(-W // TILE) * TILE
        xp = np.zeros((Hp, Wp, C + 1), np.float32)
        xp[:H, :W, :C] = x
        tiles = [(y0, x0) for y0 in range(0, Hp, TILE) for x0 in range(0, Wp, TILE)]
        mask = torch.zeros(C + 1, NTOK, device=dev)
        mask[C] = 1.0
        names = CH_EVA + [target]
        Pm = np.zeros((Hp, Wp), np.float32)
        for b0 in range(0, len(tiles), 16):
            tb = tiles[b0:b0 + 16]
            xb = np.stack([xp[y0:y0 + TILE, x0:x0 + TILE] for y0, x0 in tb])
            xb = torch.from_numpy(np.ascontiguousarray(xb.transpose(0, 3, 1, 2))).to(dev).permute(0, 2, 3, 1)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                rec, _, _ = eva(xb, marker_in=[names] * len(tb), marker_out=[[target]] * len(tb), infer_mask=mask, channel_mask=None)
            rec = rec.float()[..., 0].cpu().numpy()
            for (y0, x0), r in zip(tb, rec):
                Pm[y0:y0 + TILE, x0:x0 + TILE] = r
        return Pm[:H, :W]

    span = spans(CJT)
    ev = {f"eva__{SHORT[e]}": np.full(len(CJT), np.nan, np.float32) for e in EPI}
    same = []
    for j, s in enumerate(J_TEST):
        a, b = span[s]
        im = read_hwc(J_IMG[s], len(J_CHAN))
        mask = tifffile.imread(J_MASK[s]).astype(np.int64)
        x = eva_channels(im, jix)
        maps = {f"eva__{SHORT[e]}": eva_extra(x, EVA_EPI[e]) for e in EPI}
        same.append(float(np.abs(maps["eva__h3k27me3"] - maps["eva__phh3"]).max()))
        cm, _ = cell_means(cell_lookup(mask, CJT.object_number.values[a:b].astype(np.int64)), b - a, maps)
        for k_, v in cm.items():
            ev[k_][a:b] = v
        if j % 25 == 0:
            tick(f"EVA: {j + 1}/{len(J_TEST)} Jackson test images")
    EVA_INFO["max_abs_diff_h3k27me3_vs_phh3_output"] = float(np.max(same))
    if PJ is not None:
        for k_, v in ev.items():
            PJ[k_] = v
    print("EVA:", EVA_INFO)
except Exception:
    print("!!! EVA failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("eva")
    os.chdir(OUT)

if PJ is not None:
    PJ.to_parquet(f"{OUT}/epi_pred_jackson_test.parquet", index=False)
RUN.update(eva=EVA_INFO, failed_blocks=FAILED, minutes=(time.time() - T0) / 60, n_jackson_test_cells=int(len(CJT)),
           n_jackson_test_images=len(J_TEST), n_danenberg_cells=int(len(CD)), n_danenberg_images=len(D_ALL),
           n_danenberg_loa_images=len(D_LOA))
json.dump(RUN, open(f"{OUT}/run_info.json", "w"), indent=1, default=str)
print("failed blocks:", FAILED if FAILED else "none")
assert not {"immuvis_jackson", "immuvis_danenberg"} & set(FAILED), FAILED     # an EVA failure still leaves a usable run